# RAG Pipeline - Data Ingestion to VectorDB Pipeline

In [3]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

/var/folders/tf/dzzps07x3s54szss12f6c_8m0000gn/T/ipykernel_74991/3933654057.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader


In [4]:
### Read all PDF files in a directory

def process_all_pdf_dir(pdf_directory):

    all_docs = []
    pdf_dir = Path(pdf_directory)

    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files to process.")

    for pdf_file in pdf_files:
        print(f"\n Processing {pdf_file.name}...")
        try:
            loader = PyMuPDFLoader(str(pdf_file))
            documents = loader.load()

            for doc in documents:
                doc.metadata["source_file"] = pdf_file.name
                doc.metadata["file_type"] = "pdf"

            all_docs.extend(documents)
            print(f"Loaded {len(documents)} pages")
        
        except Exception as e:
            print(f"Error loading: {e}")
    
    print(f"\nTotal documents loaded: {len(all_docs)}")
    return all_docs

all_pdf_docs = process_all_pdf_dir("../data/")

Found 7 PDF files to process.

 Processing record linkage llm algorithms-18-00723-v2.pdf...
Loaded 14 pages

 Processing Attention-is-all-you-need-Paper.pdf...
Loaded 11 pages

 Processing deepmatcher-sigmod18.pdf...
Loaded 16 pages

 Processing 2304.12329v1.pdf...
Loaded 20 pages

 Processing 2504.15261v1.pdf...
Loaded 25 pages

 Processing Social_Media_Mining_and_Analysis.pdf...
Loaded 6 pages

 Processing probabilistic-record-linkage-using-pretrained-text-embeddings.pdf...
Loaded 12 pages

Total documents loaded: 104


In [5]:
all_pdf_docs

[Document(metadata={'producer': 'pdfTeX-1.40.25; modified using OpenPDF 1.4.2', 'creator': 'LaTeX with hyperref', 'creationdate': '2025-11-19T13:42:50+08:00', 'source': '../data/record linkage llm algorithms-18-00723-v2.pdf', 'file_path': '../data/record linkage llm algorithms-18-00723-v2.pdf', 'total_pages': 14, 'format': 'PDF 1.7', 'title': 'Efficient Record Linkage in the Age of Large Language Models: The Critical Role of Blocking', 'author': 'Nidhibahen Shah, Sreevar Patiyara, Joyanta Basak, Sartaj Sahni, Anup Mathur, Krista Park and Sanguthevar Rajasekaran', 'subject': 'Record linkage is an essential task in data integration in the fields of healthcare, law enforcement, fraud detection, transportation, biology, and supply chain management. The problem of record linkage is to cluster records from various sources such that each cluster belongs to a single entity. Scalability in record linking is limited by the large number of pairwise comparisons required. Blocking addresses this ch

In [6]:
# Text splitting function to split documents into smaller chunks
def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, 
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks.")

    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    return split_docs

In [7]:
chunks=split_documents(all_pdf_docs)
chunks

Split 104 documents into 527 chunks.

Example chunk:
Content: Academic Editor: Edward Rolando
Núñez-Valdez
Received: 30 September 2025
Revised: 3 November 2025
Accepted: 11 November 2025
Published: 16 November 2025
Citation: Shah, N.; Patiyara, S.; Basak,
J.; Sa...
Metadata: {'producer': 'pdfTeX-1.40.25; modified using OpenPDF 1.4.2', 'creator': 'LaTeX with hyperref', 'creationdate': '2025-11-19T13:42:50+08:00', 'source': '../data/record linkage llm algorithms-18-00723-v2.pdf', 'file_path': '../data/record linkage llm algorithms-18-00723-v2.pdf', 'total_pages': 14, 'format': 'PDF 1.7', 'title': 'Efficient Record Linkage in the Age of Large Language Models: The Critical Role of Blocking', 'author': 'Nidhibahen Shah, Sreevar Patiyara, Joyanta Basak, Sartaj Sahni, Anup Mathur, Krista Park and Sanguthevar Rajasekaran', 'subject': 'Record linkage is an essential task in data integration in the fields of healthcare, law enforcement, fraud detection, transportation, biology, and supply chain 

[Document(metadata={'producer': 'pdfTeX-1.40.25; modified using OpenPDF 1.4.2', 'creator': 'LaTeX with hyperref', 'creationdate': '2025-11-19T13:42:50+08:00', 'source': '../data/record linkage llm algorithms-18-00723-v2.pdf', 'file_path': '../data/record linkage llm algorithms-18-00723-v2.pdf', 'total_pages': 14, 'format': 'PDF 1.7', 'title': 'Efficient Record Linkage in the Age of Large Language Models: The Critical Role of Blocking', 'author': 'Nidhibahen Shah, Sreevar Patiyara, Joyanta Basak, Sartaj Sahni, Anup Mathur, Krista Park and Sanguthevar Rajasekaran', 'subject': 'Record linkage is an essential task in data integration in the fields of healthcare, law enforcement, fraud detection, transportation, biology, and supply chain management. The problem of record linkage is to cluster records from various sources such that each cluster belongs to a single entity. Scalability in record linking is limited by the large number of pairwise comparisons required. Blocking addresses this ch

# Embeddings and Vector Store

In [8]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [9]:
class EmbeddingManager:
    """Handling Embedded generation using Sentence Transformers"""
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """Initialize the embedding manager.

        Args:
        model_name: HuggingFace model name for sentence embeddings.
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the sentence transformer model."""
        try:
            print(f"Loaded embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(
                f"Model loaded successfully. Embedding dimension: {self.model.get_embedding_dimension()}"
            )
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts.

        Args:
        texts: List of strings to embed.

        Returns:
        Numpy array of embeddings.
        """
        if not self.model:
            raise ValueError("Model not loaded. Cannot generate embeddings.")

        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings

## Initialize Embedding Manager and generate embeddings for chunks
embedding_manager = EmbeddingManager()
embedding_manager

Loaded embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6162.90it/s]


Model loaded successfully. Embedding dimension: 384


# Vector Store using ChromaDB

In [10]:
class VectorStore:
	"""Manages documents embeddings and similarity search using ChromaDB"""

	def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
		"""Initialize the vector store.

		Args:
		collection_name: Name of the ChromaDB collection to use.
		persist_directory: Directory where ChromaDB will persist data.
		"""
		self.collection_name = collection_name
		self.persist_directory = persist_directory
		self.client = None
		self.collection = None
		self._initialize_store()

	def _initialize_store(self):
		"""Initialize the ChromaDB client and collection."""
		try:
			os.makedirs(self.persist_directory, exist_ok=True)
			self.client = chromadb.PersistentClient(path=self.persist_directory)

			self.collection = self.client.get_or_create_collection(
    			name=self.collection_name,
				metadata={
					"description": "PDF document embeddings for RAG",
					"hnsw:space": "cosine"
				}
			)

			print(f"Vector store initialized with collection: {self.collection_name}")
			print(f"Existing documents in collection: {self.collection.count()}")
		
		except Exception as e:
			print(f"Error initializing vector store: {e}")
			raise
	
	def add_documents(self, documents: List[Any], embedding: np.ndarray):
		"""Add documents and their embeddings to the vector store.

		Args:
		documents: List of Langchain document.
		embedding: Corresponding embeddings corresponding for the documents.
		"""
		if len(documents) != len(embedding):
			raise ValueError("Number of documents and embeddings must match.")
			
		print(f"Adding {len(documents)} documents to vector store...")

		ids = []
		metadatas = []
		documents_text = []
		embeddings_list = []

		for i, (doc, embedding) in enumerate(zip(documents, embedding)):
			doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
			ids.append(doc_id)

			metadata = dict(doc.metadata)
			metadata['doc_index'] = i
			metadata['content_length'] = len(doc.page_content)
			metadatas.append(metadata)

			documents_text.append(doc.page_content)

			normalized_embedding = embedding / np.linalg.norm(embedding)
			embeddings_list.append(normalized_embedding.tolist())

		try:
			self.collection.add(
				ids=ids,
				embeddings=embeddings_list,
				metadatas=metadatas,
				documents=documents_text
			)
			print(f"Successfully added {len(documents)} documents to vector store.")
			print(f"Total documents in collection: {self.collection.count()}")

		except Exception as e:
			print(f"Error adding documents to vector store: {e}")
			raise

vector_store = VectorStore()
vector_store

Vector store initialized with collection: pdf_documents
Existing documents in collection: 527


# Convert Text -> Embeddings

In [11]:
texts = [doc.page_content for doc in chunks]
texts

['Academic Editor: Edward Rolando\nNúñez-Valdez\nReceived: 30 September 2025\nRevised: 3 November 2025\nAccepted: 11 November 2025\nPublished: 16 November 2025\nCitation: Shah, N.; Patiyara, S.; Basak,\nJ.; Sahni, S.; Mathur, A.; Park, K.;\nRajasekaran, S. Efficient Record\nLinkage in the Age of Large Language\nModels: The Critical Role of Blocking.\nAlgorithms 2025, 18, 723. https://\ndoi.org/10.3390/a18110723\nCopyright: © 2025 by the authors.\nLicensee MDPI, Basel, Switzerland.\nThis article is an open access article\ndistributed under the terms and\nconditions of the Creative Commons\nAttribution (CC BY) license\n(https://creativecommons.org/\nlicenses/by/4.0/).\nalgorithms\nArticle\nEfficient Record Linkage in the Age of Large Language Models:\nThe Critical Role of Blocking\nNidhibahen Shah 1\n, Sreevar Patiyara 2, Joyanta Basak 1, Sartaj Sahni 3\n, Anup Mathur 4, Krista Park 4\nand Sanguthevar Rajasekaran 1,*\n1\nSchool of Computing, University of Connecticut, 371 Fairfield Way, 

## Generate Embedding

In [12]:
embeddings = embedding_manager.generate_embeddings(texts)

Generating embeddings for 527 texts...


Batches: 100%|██████████| 17/17 [00:04<00:00,  4.11it/s]

Generated embeddings with shape: (527, 384)


# Storing in Vector DB

In [13]:
vector_store.add_documents(chunks, embeddings)

Adding 527 documents to vector store...
Successfully added 527 documents to vector store.
Total documents in collection: 1054


# RAG Retrieval Pipeline from Vector Store

In [14]:
class RAGRetriever:
	"""Handle query based retrieval and Retrieves relevant documents from the vector store based on a query."""
	def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
		"""Initialize the RAG retriever.

		Args:
		vector_store: An instance of the VectorStore class.
		embedding_manager: An instance of the EmbeddingManager class.
		"""
		self.vector_store = vector_store
		self.embedding_manager = embedding_manager

	def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
		"""Retrieve relevant documents for a given query.

		Args:
		query: The input query string.
		top_k: The number of top relevant documents to return.
		score_threshold: The minimum score threshold for retrieved documents.

		Returns:
		A list of dictionaries containing retrieved documents and their metadata.
		"""
		print(f"Retrieving documents for query: '{query}'")
		print(f"Top K: {top_k}, Score Threshold: {score_threshold}")
		# Generate embedding for the query
		query_embedding = self.embedding_manager.generate_embeddings([query])[0]
		query_embedding = query_embedding / np.linalg.norm(query_embedding)

		# Perform similarity search in the vector store
		try:
			results = self.vector_store.collection.query(
				query_embeddings=[query_embedding.tolist()],
				n_results=top_k
			)

			retrieved_doc = []
			
			if results["documents"] and results["documents"][0]:
				documents = results["documents"][0]
				metadatas = results["metadatas"][0]
				distances = results["distances"][0]
				ids = results["ids"][0]

				for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
					similarity_score = 1 - distance  # Convert distance to similarity score (assuming cosine distance)
					if similarity_score >= score_threshold:	
						retrieved_doc.append({
							"id": doc_id,
							"content": document,
							"metadata": metadata,
							"similarity_score": similarity_score,
							"distance": distance,
							"rank": i + 1,
							
						})

				print(f"Retrieved {len(retrieved_doc)} documents(after filtering).")
			else:
				print("No documents retrieved.")
	
			return retrieved_doc
		
		except Exception as e:
			print(f"Error during retrieval: {e}")
			return []

rag_retriever = RAGRetriever(vector_store, embedding_manager)

In [15]:
rag_retriever

In [16]:
rag_retriever.retrieve("What is record linkage?")

Retrieving documents for query: 'What is record linkage?'
Top K: 5, Score Threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  3.16it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents(after filtering).


[{'id': 'doc_ef8d657e_373',
  'content': "all relevant medical information is associated with the correct individual. Record linkage or entity resolution \nis the process of connecting records from different data sources that refer to the same entity, such as a \npatient, to create a comprehensive view of that entity's information (3). Traditionally, record linkage is \nachieved through either deterministic or probabilistic methods. Deterministic record linkage relies on exact \nmatches of specific identifiers, such as names or dates of birth, ensuring high precision but struggling with \nmissing or inconsistent data. In contrast, probabilistic record linkage assigns weights to multiple identifiers, \nallowing for flexible matching despite variations, making it more effective in handling data inconsistencies \n(4,5). However, it requires careful definition of matching rules and threshold settings to determine what \nconstitutes a reliable match, which can be challenging and may vary ac

In [17]:
rag_retriever.retrieve("Mining Emotions on Plutchik’s Wheel")

Retrieving documents for query: 'Mining Emotions on Plutchik’s Wheel'
Top K: 5, Score Threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 57.68it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents(after filtering).


[{'id': 'doc_2f2f88ab_457',
  'content': 'and emotion-based hashtags. Their problem considered\n8 basic emotions of Plutchik’s wheel, however, their\nmulti-label classiﬁcation problem was completely unable\nto detect surprise, and registered low scores for fear.\nA smaller set of 4 emotions is also used by some\nresearchers [7, 19]. Although Mohammed et. al. [8]\nformulate their problem based on Plutchik’s wheel, they\nultimately boil it down to binary classiﬁcation by using\nthe one vs. other method.\nGenerally, multi-label emotion classiﬁcation suffers\nfrom either low accuracy for all classes [22] or sac-\nriﬁce the accuracy of some for the others [17, 7, 2].\nWe alleviate this issue by formulating multiple binary\nproblems, each problem consisting of a pair of polar\nopposite emotions on the axes of Plutchik’s wheel.\nAlthough each binary classiﬁcation is similar in spirit to\ndetecting positive/negative sentiment, this scheme allows\nus to learn about the speciﬁc emotions such as 

In [18]:
rag_retriever.retrieve("What is Social Media Mining")[0]['content']

Retrieving documents for query: 'What is Social Media Mining'
Top K: 5, Score Threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 25.82it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents(after filtering).


'Tweets thus contain a treasure trove of information that\ncan offer clues about users’ opinions, thoughts, and\nfeelings on a variety of topics from politics to restaurants\nto even their mental health.\nThe plethora of information embedded in these tweets\nhas attracted signiﬁcant attention in their mining and\nanalysis. A large body of work has focused on detecting\nand classifying the sentiment and/or polarity of the\ntweets [24]. In binary sentiment analysis, tweets are\ngrouped according to positive and negative polarities,\nwhereas in multi-class analysis they are grouped into\nmore than two classes according to the strength of the\nembedded sentiment. Tweets can also be mined for\naffective information (moods, emotions, and feelings) of\nthe tweeters. Emotion mining thus can be viewed as a\ndeeper, more advanced form of sentiment analysis [13].\nThis detailed, granular information can support a range\nof applications such as targeted advertising, recommend-'

# Integrating VectorDB Context Pipeline to LLM (Augmentation)

In [19]:
### Simple RAG pipeline with Grok LLM

from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

llm = ChatGroq(groq_api_key= os.getenv("GROQ_API_KEY"), model_name="openai/gpt-oss-safeguard-20b", temperature=0.1, max_tokens=1024)

## Simple RAG function: retrieve + augment + generate answer

def rag_simple(query, retriever, llm, top_k=3):
	#retrieve context

	results = retriever.retrieve(query, top_k=top_k)
	context = "\n\n".join([doc['content'] for doc in results]) if results else "No  documents found."

	if not context:
		return "No relevant context found."

	#generate answer using LLM
	prompt = f"""Use the following context to answer the question:\n\n 
	Context:\n{context}\n\n
	Question: {query}\n\n
	Answer:"""

	response = llm.invoke([prompt.format(context=context, query=query)])
	return response.content

In [21]:
answer = rag_simple("What is bitcoin?", rag_retriever, llm)
print(f"Answer:\n{answer}")

Retrieving documents for query: 'What is bitcoin?'
Top K: 3, Score Threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  3.20it/s]


Generated embeddings with shape: (1, 384)
Retrieved 3 documents(after filtering).
Answer:
Bitcoin is a decentralized digital currency (cryptocurrency) that operates on a peer‑to‑peer network using blockchain technology. It was introduced in 2009 by an anonymous entity known as Satoshi Nakamoto. Transactions are recorded on a public ledger (the blockchain) and validated by network participants (miners) through cryptographic proof‑of‑work. Bitcoin can be used for online payments, investment, or as a store of value, and its supply is capped at 21 million coins.


In [22]:
answer = rag_simple("Who is Dhruv?", rag_retriever, llm)
print(f"Answer:\n{answer}")

Retrieving documents for query: 'Who is Dhruv?'
Top K: 3, Score Threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  2.10it/s]


Generated embeddings with shape: (1, 384)
Retrieved 3 documents(after filtering).
Answer:
The provided excerpt does not contain any information about a person named Dhruv, so I’m unable to identify who Dhruv is based on this context.


# Enhance RAG
**RAG pipeloine with extra feature:**

- Return answer
- return source
- return confidence score
- optionally full context

In [23]:
def rag_advanced(query, retriever, llm, top_k=3, min_score=0.2, return_context=False):
	results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)

	if not results:
		return {"answer": "No relevant documents found.", "sources": [], "confidence": 0.0, "context": ''}

	context = "\n\n".join([doc['content'] for doc in results])

	sources = [{
		'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
		'page': doc['metadata'].get('page', 'unknown'),
		'score': doc['similarity_score'],
		'preview': doc['content'][:200] + '...'
	} for doc in results]

	confidence = max(doc['similarity_score'] for doc in results)

	prompt = f"Use the following context to answer the question concisely:\n\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"

	response = llm.invoke([prompt.format(context=context, query=query)])

	output = {
		"answer": response.content,
		"sources": sources,
		"confidence": confidence
	}
	if return_context:
		output["context"] = context
	return output

result = rag_advanced("What is bitcoin?", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context:", result['context'][:200])

Retrieving documents for query: 'What is bitcoin?'
Top K: 3, Score Threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:03<00:00,  3.25s/it]


Generated embeddings with shape: (1, 384)
Retrieved 3 documents(after filtering).
Answer: Bitcoin is a decentralized digital currency (cryptocurrency) that operates on a peer‑to‑peer network using blockchain technology. It allows users to send and receive payments without a central authority, with transactions verified by miners and recorded in a public ledger.
Sources: [{'source': 'Social_Media_Mining_and_Analysis.pdf', 'page': 0, 'score': 0.15782439708709717, 'preview': 'Tweets thus contain a treasure trove of information that\ncan offer clues about users’ opinions, thoughts, and\nfeelings on a variety of topics from politics to restaurants\nto even their mental health.\n...'}, {'source': 'Social_Media_Mining_and_Analysis.pdf', 'page': 0, 'score': 0.15782439708709717, 'preview': 'Tweets thus contain a treasure trove of information that\ncan offer clues about users’ opinions, thoughts, and\nfeelings on a variety of topics from politics to restaurants\nto even their mental health.\n...

# More Advanced RAG Pipeline

- Streaming
- Citations
- History
- Summarization

In [24]:
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]
            # Streaming answer simulation
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
            # if stream:
            #     print("Streaming answer:")
            #     for i in range(0, len(prompt), 80):
            #         print(prompt[i:i+80], end='', flush=True)
            #         time.sleep(0.05)
            #     print()
            response = self.llm.invoke([prompt.format(context=context, question=question)])
            answer = response.content

        # Add citations to answer
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("What is Blocking model performance ", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print()
print("Summary:", result['summary'])
print()
print("History:", result['history'][-1])

Retrieving documents for query: 'What is Blocking model performance '
Top K: 3, Score Threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:03<00:00,  3.09s/it]


Generated embeddings with shape: (1, 384)
Retrieved 3 documents(after filtering).

Final Answer: The blocking model delivers a marked boost in efficiency over traditional rule‑based blocking, but at the cost of missing a few true matches (i.e., a slight drop in recall).

Citations:
[1] 2304.12329v1.pdf (page 4)
[2] 2304.12329v1.pdf (page 4)
[3] 2504.15261v1.pdf (page 16)

Summary: The blocking model significantly boosts efficiency compared to traditional rule‑based blocking. However, this improvement comes with a small trade‑off: it misses a few true matches, slightly lowering recall.

History: {'question': 'What is Blocking model performance ', 'answer': 'The blocking model delivers a marked boost in efficiency over traditional rule‑based blocking, but at the cost of missing a few true matches (i.e., a slight drop in recall).', 'sources': [{'source': '2304.12329v1.pdf', 'page': 4, 'score': 0.7128975987434387, 'preview': 'execution time for each model.\nBlocking. Given the vectorized e